**Import libraries**

In [1]:
!pip install awswrangler
!pip install --upgrade google-api-python-client oauth2client
!pip install gspread

In [2]:
from __future__ import print_function

import argparse
import os, sys
import awswrangler as wr

sys.path.append("/home/ec2-user/SageMaker/ml-demand-forecasting")


# standard libraries
import pandas as pd
import time
import numpy as np
import random
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
from src.io.loaders import load_external_data
from pomelo.utils import Hal
from config.project_config import *


from src.queries.query_data_c_offline import query_data
from src.io.load_data_c_offline import load_data
from src.feature.add_covid_offline import add_covid
from src.preprocessing.data_cleaning import *
from src.preprocessing.prep_offline_data_raw import *
from src.io.loaders import upload_file_to_s3
from src.preprocessing.offline_processing_transform import transform_data



# for model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    max_error,
)
import pickle
import s3fs

import warnings
from sklearn.exceptions import DataConversionWarning
from pandas.core.common import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings(action="ignore", category=DataConversionWarning)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 10000)
pd.set_option("max_colwidth", 10000)

from datetime import date
today = date.today()
today = today.strftime("%d%m%Y")
today = '20072022'

temp_data_save_path = f'../../temp/data_prep/{today}/offline/'
if not os.path.exists(temp_data_save_path):
    os.makedirs(temp_data_save_path)

S3_FOLDER_PATH = f'data_science/dfm/offline_clothing_v2/{today}'

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/boto3/compat.py:88: PythonDeprecationWarning: Boto3 will no longer support Python 3.6 starting May 30, 2022. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.7 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/psycopg2/__init__.py:144: UserWarning: The psycopg2 wheel package will be renamed from release 2.8; in order to keep installing from binary please use "pip install psycopg2-binary" instead. For details see: <http://initd.org/psycopg/docs/install.html#binary-install-from-pypi>.
  """)
/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/boto3/compat.py:88: PythonDeprecationWarning: Boto3 will no longer support Python 3.6 starting May 30, 2022. To contin

In [3]:
# # Get a SageMaker-compatible role used by this Notebook Instance.

# import sagemaker
# from sagemaker import get_execution_role

# sagemaker_session = sagemaker.Session()
# role = get_execution_role()

# 1. Query Data 

In [4]:
%%time
# Query raw data
raw = query_data()

query_data took: 0:01:28.131563
CPU times: user 13.3 s, sys: 3.31 s, total: 16.6 s
Wall time: 1min 28s


In [5]:
raw

,id_product_attribute,id_product,henry_id_product_attribute,henry_id_product,date_released,product_cost_usd,size_range,size,brand,product_line,sub_product_line,old_henry_category_1,old_henry_category_2,old_henry_category_3,henry_category_1,henry_category_2,henry_category_3,simple_color,color,style,sleeve,pattern,sleevestyle,neckline,shape,rise,original_price_usd,fabric_custom_name,hscode_id_fabric_name,release_collection_name,id_store,store_name,id_shop,id_shop_name,warehouse,first_available_date,first_available_month,first_available_dow,day_number,first_available_year,first_week_of_month,last_week_of_month,giveaway,max_transaction_date,min_transaction_date,net_units_sold_week1,net_units_sold_week2,net_units_sold_week3,net_units_sold_week4,net_units_sold_week5,net_units_sold_week6,net_units_sold_week7,gross_revenue_usd_week1,gross_revenue_usd_week2,gross_revenue_usd_week3,gross_revenue_usd_week4,gross_revenue_usd_week5,gross_revenue_usd_week6,gross_revenue_usd_week7,revenue_usd_week1,revenue_usd_week2,revenue_usd_week3,revenue_usd_week4,revenue_usd_week5,revenue_usd_week6,revenue_usd_week7,item_discount_usd_week1,item_discount_usd_week2,item_discount_usd_week3,item_discount_usd_week4,item_discount_usd_week5,item_discount_usd_week6,item_discount_usd_week7,voucher_discount_usd_week1,voucher_discount_usd_week2,voucher_discount_usd_week3,voucher_discount_usd_week4,voucher_discount_usd_week5,voucher_discount_usd_week6,voucher_discount_usd_week7,is_mega_campaign_order_week1,is_mega_campaign_order_week2,is_mega_campaign_order_week3,is_mega_campaign_order_week4,is_mega_campaign_order_week5,is_mega_campaign_order_week6,is_mega_campaign_order_week7
0,131125,38800,172312,78883,2021-01-06,6.040277,"XXL,XS,XL,S,M,L",S,Pomelo,Essential,Basic,Tops,Tees,Sleeveless,Tops,Tanks,-,White,White,Tank,Sleeveless,NoPattern,RegularSleeve,CrewNeck,<NA>,<NA>,18.64864,Knit,Cotton,Jan21 : Essential Basic,351,The Mall Ngamwongwan,1,TH,TH,2021-01-13,January,Wednesday,13,2021,0,0,1,2021-03-28,2021-01-16,0,1,0,0,0,0,1,0.000000,22.934988,0.000000,0.000000,0.000000,0.000000,23.011477,0.000000,22.934988,0.000000,0.000000,0.000000,0.000000,18.409181,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0,0,0,0,0,0,0
1,129899,38352,175703,79822,2020-12-23,5.390000,"XXL,XS,XL,S,M,L",S,Pomelo,Core Products,Weekend,Outerwears,Jackets,-,Tops,Blouses,-,Nude,Beige,Training,LongSleeve,<NA>,PuffSleeve,<NA>,<NA>,<NA>,15.94594,Knit,Polyester,Dec20 - Drop 7 - Reprise,371,Siam Center,1,TH,TH,2020-12-23,December,Wednesday,23,2020,0,1,7,2021-04-17,2020-12-24,4,0,0,0,0,1,0,78.521676,0.000000,0.000000,0.000000,0.000000,19.716287,0.000000,75.381341,0.000000,0.000000,0.000000,0.000000,19.716287,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.140335,0.0,0.0,0.0,0.000000,0.0,0.000000,0,0,0,0,0,0,0
2,132373,39220,178453,80688,2021-01-06,2.820000,"XXL,XS,XL,S,M,L",L,Pomelo,Campaign,Collaboration,Tops,Camis and Tanks,Tanks,Tops,Tanks,-,Black,Black,OneShoulder,Sleeveless,NoPattern,RegularSleeve,Asymmetric,<NA>,<NA>,7.83783,Knit,Polyester,Jan21 : Play,251,EmQuartier,1,TH,TH,2021-01-13,January,Wednesday,13,2021,0,0,5,2021-02-11,2021-02-11,0,0,0,0,0,1,0,0.000000,0.000000,0.000000,0.000000,0.000000,9.708660,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.189438,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0,0,0,0,0,0,0
3,132373,39220,178453,80688,2021-01-06,2.820000,"XXL,XS,XL,S,M,L",L,Pomelo,Campaign,Collaboration,Tops,Camis and Tanks,Tanks,Tops,Tanks,-,Black,Black,OneShoulder,Sleeveless,NoPattern,RegularSleeve,Asymmetric,<NA>,<NA>,7.83783,Knit,Polyester,Jan21 : Play,381,Fashion Island,1,TH,TH,2021-01-14,January,Thursday,14,2021,0,0,5,2021-04-03,2021-01-16,0,1,0,0,1,0,1,0.000000,9.639343,0.000000,0.000000,9.660536,0.000000,9.663355,0.000000,9.639343,0.000000,0.000000,7.245402,0.000000,9.663355,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,2.415134,0.0,0.000000,0,0,0,0,0,0,0
4,194837,62233,219166,94882,2022-01-19,5.142124,"XXL,XS,XL,S,M,L"

# 2. Data preparation 

In [7]:
%%time
# unpivot 'net_units_sold','gross_revenue_usd','revenue_usd','is_mega_campaign_order','item_discount_usd','voucher_discount_usd'
unpivot_raw = (
    pd.wide_to_long(
        raw.reset_index(),
        stubnames=[
            "net_units_sold",
            "gross_revenue_usd",
            "revenue_usd",
            "is_mega_campaign_order",
            "item_discount_usd",
            "voucher_discount_usd",
        ],
        i="index",
        j="week_id",
        sep="_",
        suffix="\w+",
    )
    .reset_index(level=1)
    .reset_index(drop=True)
)

CPU times: user 24 s, sys: 2.65 s, total: 26.7 s
Wall time: 26.7 s


In [5]:
unpivot_raw

,week_id,original_price_usd,first_available_date,store_name,day_number,id_store,last_week_of_month,fabric_custom_name,giveaway,pattern,max_transaction_date,first_available_year,hscode_id_fabric_name,old_henry_category_3,id_shop,henry_id_product,old_henry_category_2,henry_category_1,style,brand,henry_category_2,product_line,sub_product_line,henry_category_3,date_released,shape,first_week_of_month,color,id_product,id_shop_name,sleevestyle,first_available_month,size,simple_color,product_cost_usd,release_collection_name,first_available_dow,size_range,old_henry_category_1,henry_id_product_attribute,neckline,warehouse,id_product_attribute,min_transaction_date,rise,sleeve,net_units_sold,gross_revenue_usd,revenue_usd,is_mega_campaign_order,item_discount_usd,voucher_discount_usd
0,week1,18.64864,2021-01-13,The Mall Ngamwongwan,13,351,0,Knit,1.0,NoPattern,2021-03-28,2021,Cotton,Sleeveless,1,78883,Tees,Tops,Tank,Pomelo,Tanks,Essential,Basic,-,2021-01-06,NaN,0,White,38800,TH,RegularSleeve,January,S,White,6.040277,Jan21 : Essential Basic,Wednesday,"XXL,XS,XL,S,M,L",Tops,172312,CrewNeck,TH,131125,2021-01-16,NaN,Sleeveless,0,0.000000,0.000000,0,0.0,0.000000
1,week1,15.94594,2020-12-23,Siam Center,23,371,1,Knit,7.0,NaN,2021-04-17,2020,Polyester,-,1,79822,Jackets,Tops,Training,Pomelo,Blouses,Core Products,Weekend,-,2020-12-23,NaN,0,Beige,38352,TH,PuffSleeve,December,S,Nude,5.390000,Dec20 - Drop 7 - Reprise,Wednesday,"XXL,XS,XL,S,M,L",Outerwears,175703,NaN,TH,129899,2020-12-24,NaN,LongSleeve,4,78.521676,75.381341,0,0.0,3.140335
2,week1,7.83783,2021-01-13,EmQuartier,13,251,0,Knit,5.0,NoPattern,2021-02-11,2021,Polyester,Tanks,1,80688,Camis and Tanks,Tops,OneShoulder,Pomelo,Tanks,Campaign,Collaboration,-,2021-01-06,NaN,0,Black,39220,TH,RegularSleeve,January,L,Black,2.820000,Jan21 : Play,Wednesday,"XXL,XS,XL,S,M,L",Tops,178453,Asymmetric,TH,132373,2021-02-11,NaN,Sleeveless,0,0.000000,0.000000,0,0.0,0.000000
3,week1,7.83783,2021-01-14,Fashion Island,14,381,0,Knit,5.0,NoPattern,2021-04-03,2021,Polyester,Tanks,1,80688,Camis and Tanks,Tops,OneShoulder,Pomelo,Tanks,Campaign,Collaboration,-,2021-01-06,NaN,0,Black,39220,TH,RegularSleeve,January,L,Black,2.820000,Jan21 : Play,Thursday,"XXL,XS,XL,S,M,L",Tops,178453,Asymmetric,TH,132373,2021-01-16,NaN,Sleeveless,0,0.000000,0.000000,0,0.0,0.000000
4,week1,13.48648,2022-01-17,Mega Bangna,17,211,0,Woven,0.0,None,2022-02-02,2022,Cotton,Sleeveless,1,94882,Tees,Tops,None,Pomelo,Tees,Campaign,Collaboration,-,2022-01-19,NaN,0,Brown,62233,TH,None,January,XS,Brown,5.142124,Jan22 : Pantone drop 2,Monday,"XXL,XS,XL,S,M,L",Tops,219166,None,TH,194837,2022-01-18,NaN,None,1,15.154965,15.154965,0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3975589,week7,29.45945,2020-10-29,Mega Bangna,29,211,1,Woven,9.0,NaN,2020-12-13,2020,Polyester,-,1,78378,Dresses - Mini,Dresses,Aline,Pomelo,Mini,Campaign,Collection,-,2020-10-28,NaN,0,White,37552,TH,PuffSleeve,October,XS,White,9.882192,Oct20 Drop8 Pomelo Couture (Wedding guest),Thursday,"XXL,XS,XL,S,M,L",Dresses,170238,Boat,TH,126611,2020-10-31,NaN,HalfSleeve,1,36.238508,29.589241,0,0.0,0.000000
3975590,week7,9.43243,2022-02-19,Central Rama 9,19,311,0,Knit,0.0,None,2022-03-29,2022,Rayon,Short Sleeve,1,108525,Tees,Tops,None,Pomelo,Blouses,Essential,Basic,-,2022-02-15,NaN,0,Pink,67504,TH,None,February,L,Pink,4.310000,Feb22: RENECO Top,Saturday,"XXS/XS,XXS,XS,XL,S/M,S,M,L/XL,L",Tops,255022,None,TH,209267,2022-03-09,NaN,None,1,10.421176,6.549055,0,0.0,0.000000
3975591,week7,26.75675,2019-02-01,Icon Siam,1,11,0,Woven,1.0,NoPattern,2019-03-16,2019,Polyester,-,1,61011,Skorts,Bottoms,FrontButtoned,Pomelo,Skorts,Core Products,Weekend,-,2019-02-01,Fitted,1,Pink,21902,TH,NaN,February,XXL,Pink,8.800000,Feb19 Drop1,Friday,"XXL,XS,XL,S,M,L",Bottoms,92453,NaN,TH,54045,2019-03-16,HighWaisted,NaN,1,24.989321,21.240923,0,0.0,0.00

In [ ]:
Prep_offline_data = prep_offline_data(unpivot_raw=unpivot_raw)
raw6 = Prep_offline_data.run(unpivot_raw)

# 3. back fill new categories

In [9]:
len(raw6[raw6['henry_category_1'].isna()])/len(raw6) 

0.060440370978865496

In [10]:
for cat in ['1', '2', '3']:
    print(f'back fill henry category {cat}')
    raw6 = fill_new_categories(raw6, cat)
    print('--------')

back fill henry category 1
Unique categories before mapping: ['Tops' 'Bottoms' 'Dresses' 'Outerwears' nan 'Jumpsuits & Rompers'
 'ACTIVEWEAR' 'Swimwear-Bottoms' 'Swimwear-Tops' 'Swimsuits']
Number of NA rows before mapping missing categories: (226982, 63)
Unique categories after mapping: ['Tops' 'Bottoms' 'Dresses' 'Outerwears' nan 'Jumpsuits & Rompers'
 'ACTIVEWEAR' 'Swimwear-Bottoms' 'Swimwear-Tops' 'Swimsuits']
Number of NA rows after mapping missing categories: (118363, 64)
--------
back fill henry category 2
Unique categories before mapping: ['Camis' 'Shorts' 'Mini' 'Cloaks & Cover Ups' 'Midi' 'Bustiers' nan
 'Blouses' 'Shirts' 'Pants' 'Jackets' 'Tanks' 'Knee Length' 'Skirts'
 'Skorts' 'Cardigans' 'Knitted Tops' 'Jumpsuits' 'Tees' 'Jeans' '-'
 'Bodysuits' 'Maxi' 'Blazers' 'Rompers' 'Bikini Bottoms'
 'Sweatshirts /Hoodies' 'Sports Bra' 'Vests' 'Bikini Tops' 'Overalls'
 'One Piece' 'Leggings' 'Coats' 'Legging - Knee length' 'Sweat pants']
Number of NA rows before mapping missing cat

In [11]:
len(raw6[raw6['henry_category_1'].isna()])/len(raw6) 

0.03151749314999188

In [12]:
''' Drop NA in Category 1 '''
raw_final = raw6[~raw6.henry_category_1.isna()]

In [13]:
raw_final.to_parquet(temp_data_save_path + 'raw_final.parquet')

In [14]:
# upload to s3
upload_file_to_s3(
    local_path=temp_data_save_path + 'raw_final.parquet',
    s3_key=f'{S3_FOLDER_PATH}/raw_final.parquet',
    bucket_name=S3_BUCKET
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/boto3/compat.py:88: PythonDeprecationWarning: Boto3 will no longer support Python 3.6 starting May 30, 2022. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.7 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


# 4. Transform Data 

In [4]:
raw_final = pd.read_parquet(temp_data_save_path + 'raw_final.parquet')

In [5]:
raw_final

,week_id,first_week_of_month,count_products_in_group,sleeve,original_price_usd,release_collection_name,size,brand,style,giveaway,product_line,neckline,henry_category_1,store_name,simple_color,henry_category_2,sub_product_line,min_transaction_date,days_in_stock_lt,color,shape,hscode_id_fabric_name,day_number,date_released,first_available_year,henry_category_3,old_henry_category_3,id_product_attribute,last_week_of_month,henry_id_product,id_store,first_available_date,product_cost_usd,covid_lockdown,sleevestyle,warehouse,id_product,pattern,old_henry_category_2,old_henry_category_1,max_transaction_date,rise,henry_id_product_attribute,fabric_custom_name,id_shop_name,inventory_per_store,first_available_month,first_available_dow,id_shop,net_units_sold,gross_revenue_usd,revenue_usd,is_mega_campaign_order,item_discount_usd,voucher_discount_usd,start_date_weekly,covid_cases,year_week_id,month,year,traffic_dist,retail_mkt_spend_dist
0,week1,0,1.0,Sleeveless,31.14285,Premium Collection_September 2019,M,Pomelo,Tank,3.0,Core Products,Square,Tops,Central Rama 9,Grey,Camis,Weekend,2019-10-02,18.0,Light Grey,None,Polyester,24,2019-09-25,2019,-,Camisoles,81467,1,66737,311,2019-09-24,9.290000,0,SpaghettiStrap,TH,27517,Stripes,Camis and Tanks,Tops,2019-11-01,None,119635,Woven,TH,6.0,September,Tuesday,1,0,0.000000,0.000000,0,0.00000,0.00000,2019-09-24,NaN,201939,September,2019,0.000000,7.138625
1,week1,0,NaN,None,25.42857,Aug19 Drop5,L,Pomelo,Chinos,0.0,Core Products,None,Bottoms,Central Phuket,Pink,Shorts,Workwear,2019-12-16,46.0,Pink,WideLeg,Polyester,28,2019-08-09,2019,-,-,73405,1,65158,301,2019-11-28,6.930000,0,None,TH,25807,Stripes,Shorts,Bottoms,2019-12-16,HighWaisted,111786,Woven,TH,NaN,November,Thursday,1,0,0.000000,0.000000,0,0.00000,0.00000,2019-11-28,NaN,201948,November,2019,37.893742,13.919672
2,week1,0,14.0,None,28.54285,Apr22: Workwear [april style] Drop 2,XS,Pomelo,None,0.0,Core Products,None,Dresses,Terminal 21,Pink,Mini,Workwear,2022-04-27,17.0,Light Pink,None,Tencel,25,2022-04-25,2022,-,-,208584,1,106142,361,2022-04-25,16.084528,0,None,TH,67327,None,Dresses - Mini,Dresses,2022-05-13,None,246159,Woven,TH,58.0,April,Monday,1,1,29.077469,19.731511,0,0.00000,0.00000,2022-04-25,NaN,202218,April,2022,20.988889,0.000000
3,week1,1,1.0,LongSleeve,34.00000,June19 Drop 11,L,Pomelo,Kimono & Wraps,2.0,Campaign,None,Outerwears,EmQuartier,Multi,Cloaks & Cover Ups,Collection,2019-07-14,21.0,Print,None,Rayon,1,2019-06-26,2019,-,-,63721,0,63087,251,2019-07-01,12.600000,0,None,TH,24013,Floral,Kimonos & Wraps,Outerwears,2019-07-14,None,101036,Woven,TH,1.0,July,Monday,1,0,0.000000,0.000000,0,0.00000,0.00000,2019-07-01,NaN,201927,July,2019,31.150584,0.208688
4,week1,0,2.0,None,42.82857,Jan22: CNY Inhouse,S,Pomelo,None,43.0,Campaign,None,Dresses,SG 313 Somerset,Red,Midi,Collection,2022-01-12,34.0,Red,None,Polyester,31,2022-01-05,2021,-,-,185404,1,97616,12,2021-12-31,16.170000,0,None,MY,58475,None,Dresses - Midi,Dresses,2022-01-31,None,226004,Woven,SG,25.0,December,Friday,2,0,0.000000,0.000000,0,0.00000,0.00000,2021-12-31,NaN,202201,December,2021,12.958217,0.588436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3755464,week7,0,3.0,None,22.57142,Aug20 Drop3 Barbie + Denim China,25,Pomelo,FrontButtoned,12.0,Essential,None,Bottoms,Zpell,Black,Skirts,Denims,2020-09-28,55.0,Black,Fitted,Cotton,19,2020-08-19,2020,Mini Skirts,Mini Skirts,120526,0,75615,331,2020-08-19,9.757309,0,None,TH,36005,NoPattern,Skirts,Bottoms,2020-10-12,HighWaisted,159120,Woven,TH,48.0,August,Wednesday,1,0,0.000000,0.000000,0,0.00000,0.00000,2020-09-30,NaN,202040,September,2020,6.603529,1.015745
3755466,week7,0,1.0,None,22.82857,SEP21 : Play trend,S,Pomelo,None,2.0,Core Products,None,Outerwears,Terminal 21,Pink,Cardigans,Workwear,2021-10-09,15.0,Pink,None,Cotton,27,2021-09-27,2021,-,-,1

In [8]:
master_join_offline =  transform_data(raw_final)

transform_data took: 0:00:50.497263


In [9]:
master_join_offline['spl_cluster'].value_counts()

Weekend_B          601798
Weekend_C          542975
Collection_B       259128
Weekend_A          237846
Collection_C       232881
Basic_C            142145
Basic_B            137292
Workwear_B         133517
Denims_A           129256
Weekend_D          122144
Collection_A       118569
Basic_A            118515
Collaboration_C    109315
Workwear_C         103895
Collaboration_B    101889
Workwear_A          84665
Collaboration_A     75807
Collaboration_D     61873
Denims_B            57282
Workwear_D          56838
Collection_D        55956
Basic_D             52817
Weekend_-           19572
Collaboration_-      7609
Collection_-         7483
Basic_-              7084
Workwear_-           4067
Denims_-             3248
Name: spl_cluster, dtype: int64

In [10]:
master_join_offline = clean_data(master_join_offline)

clean_data took: 0:05:24.323685


In [11]:
master_join_offline

,henry_id_product_attribute,henry_id_product,id_product_attribute,id_product,product_cost_usd,original_price_usd,size,product_line,sub_product_line,henry_category_1,henry_category_2,henry_category_3,old_henry_category_1,old_henry_category_2,old_henry_category_3,simple_color,fabric_custom_name,hscode_id_fabric_name,is_mega_campaign_order,giveaway,cluster,adjusted_net_units_sold,first_available_month,first_available_dow,first_available_year,discount_utilization,item_discount_percent,voucher_discount_percent,first_week_of_month,last_week_of_month,color,warehouse,style,sleeve,pattern,sleevestyle,neckline,shape,rise,release_collection_name,covid_lockdown,count_products_in_group,inventory_per_store,week_id,id_shop_name,traffic_dist,retail_mkt_spend_dist,date_released,first_available_date,store_name,id_store,spl_cluster
0,119635,66737,81467,27517,9.290000,31.14285,M,core_products,weekend,tops,camis,NaN,Tops,Camis and Tanks,Camisoles,grey,woven,polyester,0,1,C,0,september,tuesday,2019,0.000000,0.0,0.00,0,1,light_grey,TH,tank,sleeveless,stripes,spaghettistrap,square,NaN,NaN,Premium Collection_September 2019,0,1.0,6.0,week1,TH,0.000000,7.138625,2019-09-25,2019-09-24,Central Rama 9,311,Weekend_C
1,111786,65158,73405,25807,6.930000,25.42857,L,core_products,workwear,bottoms,shorts,NaN,Bottoms,Shorts,-,pink,woven,polyester,0,0,D,0,november,thursday,2019,0.000000,0.0,0.00,0,1,pink,TH,chinos,NaN,stripes,NaN,NaN,wideleg,highwaisted,Aug19 Drop5,0,0.0,0.0,week1,TH,37.893742,13.919672,2019-08-09,2019-11-28,Central Phuket,301,Workwear_D
2,246159,106142,208584,67327,16.084528,28.54285,XS,core_products,workwear,dresses,mini,NaN,Dresses,Dresses,Dresses - Mini,pink,woven,tencel,0,0,C,1,april,monday,2022,0.321416,0.0,0.00,0,1,light_pink,TH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Apr22: Workwear [april style] Drop 2,0,14.0,58.0,week1,TH,20.988889,0.000000,2022-04-25,2022-04-25,Terminal 21,361,Workwear_C
3,101036,63087,63721,24013,12.600000,34.00000,L,campaign,collection,outerwears,cloaks_&_cover_ups,NaN,Outerwears,Kimonos & Wraps,-,multi,woven,rayon,0,1,B,0,july,monday,2019,0.000000,0.0,0.00,1,0,print,TH,kimono_&_wraps,longsleeve,floral,NaN,NaN,NaN,NaN,June19 Drop 11,0,1.0,1.0,week1,TH,31.150584,0.208688,2019-06-26,2019-07-01,EmQuartier,251,Collection_B
4,226004,97616,185404,58475,16.170000,42.82857,S,campaign,collection,dresses,midi,NaN,Dresses,Dresses,Dresses - Midi,red,woven,polyester,0,1,B,0,december,friday,2021,0.000000,0.0,0.00,0,1,red,MY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Jan22: CNY Inhouse,0,2.0,25.0,week1,SG,12.958217,0.588436,2022-01-05,2021-12-31,SG 313 Somerset,12,Collection_B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3637102,159120,75615,120526,36005,9.757309,22.57142,25,essential,denims,bottoms,skirts,mini_skirts,Bottoms,Skirts,Mini Skirts,black,woven,cotton,0,1,A,0,august,wednesday,2020,0.000000,0.0,0.00,0,0,black,TH,frontbuttoned,NaN,NaN,NaN,NaN,fitted,highwaisted,Aug20 Drop3 Barbie + Denim China,0,3.0,48.0,week7,TH,6.603529,1.015745,2020-08-19,2020-08-19,Zpell,331,Denims_A
3637103,205062,89155,170182,52446,7.420000,22.82857,S,core_products,workwear,outerwears,cardigans,NaN,Outerwears,Cardigans,-,pink,knit,cotton,0,1,C,0,september,monday,2021,0.000000,0.0,0.00,0,1,pink,TH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SEP21 : Play trend,0,1.0,10.0,week7,TH,2.937026,0.036270,2021-09-27,2021-09-27,Terminal 21,361,Workwear_C
3637104,115431,65453,74634,26059,12.740000,36.85714,XS,campaign,collection,dresses,midi,NaN,Dresses,Dresses,Dresses - Midi,pink,woven,polyester,1,1,A,1,august,wednesday,2019,0.200000,0.2,0.00,0,1,pink,TH,line,shortsleeve,floral,puffsleeve,v,NaN,NaN,Fall Campaign 19 - Phase 1,0,2.0,9.0,week7,TH,15.487274,0.686554,2019-08-28,2019-08-28,Mega Bangna,211,Collection_A
3637105,177370,80381,131641,38980,8.520000,31.14285,XS,core_products,weekend,dresses,mini,NaN,Dresses,Dresses,Dresses - Mini,white

In [12]:
today = '11082022'

temp_data_save_path = f'../../temp/data_prep/{today}/offline/'
if not os.path.exists(temp_data_save_path):
    os.makedirs(temp_data_save_path)

S3_FOLDER_PATH = f'data_science/dfm/offline_clothing_v2/{today}'

In [13]:
master_join_offline.to_parquet(temp_data_save_path + 'master_join_offline.parquet')

In [14]:
upload_file_to_s3(
    local_path=temp_data_save_path + 'master_join_offline.parquet',
    s3_key=f'{S3_FOLDER_PATH}/master_join_offline.parquet',
    bucket_name=S3_BUCKET
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.6/site-packages/boto3/compat.py:88: PythonDeprecationWarning: Boto3 will no longer support Python 3.6 starting May 30, 2022. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.7 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [15]:
master_join_offline.isna().sum()

henry_id_product_attribute          0
henry_id_product                    0
id_product_attribute                0
id_product                          0
product_cost_usd                    0
original_price_usd                  0
size                                0
product_line                        0
sub_product_line                    0
henry_category_1                    0
henry_category_2                 1344
henry_category_3              3011040
old_henry_category_1                0
old_henry_category_2                0
old_henry_category_3                0
simple_color                        0
fabric_custom_name                  0
hscode_id_fabric_name               0
is_mega_campaign_order              0
giveaway                            0
cluster                         49517
adjusted_net_units_sold             0
first_available_month               0
first_available_dow                 0
first_available_year                0
discount_utilization                0
item_discoun